# 2023 Travel Bookings: Data Screening, Cleaning & Transformation

**Input:** `raw/Bookings_2023 (1).csv`  
**Output:** `clean/bookings_cleaned_python_2023.csv`

**Workflow**

1. Load the raw data
2. Screen it (structure, missing values, duplicates, inconsistent labels, integrity checks)
3. Clean it (standardise `Status`, rename a column, reformat dates)
4. Transform it (add `HeadCount`, `TotalPrice`, `ActualPaidPrice`)
5. Validate the result
6. Export the cleaned file

> Run this notebook from the `notebook/` folder so the relative paths below resolve correctly.

In [ ]:
import pandas as pd
import numpy as np

# ---- Paths (relative to the notebook/ folder) ----
RAW_PATH = "../raw/Bookings_2023 (1).csv"
CLEAN_PATH = "../clean/bookings_cleaned_python_2023.csv"

# ---- Discount formula switch ----
# False -> reproduces the existing cleaned file: TotalPrice * (1 - DiscountPct / 100)
# True  -> treats DiscountPct as a fraction (0.15 = 15%): TotalPrice * (1 - DiscountPct)
# See the note in section 4 before changing this.
DISCOUNT_AS_FRACTION = False

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## 1. Load the raw data

In [ ]:
raw = pd.read_csv(RAW_PATH)
print(f"Shape: {raw.shape[0]:,} rows x {raw.shape[1]} columns")
raw.head()

## 2. Data screening

Screening is done on the untouched raw data. Nothing is changed in this section.

### 2.1 Structure and data types

In [ ]:
raw.info()

### 2.2 Missing values

In [ ]:
missing = pd.DataFrame({
    "missing": raw.isna().sum(),
    "missing_pct": (raw.isna().mean() * 100).round(1),
})
missing[missing["missing"] > 0]

### 2.3 Duplicates

In [ ]:
print("Fully duplicated rows:", raw.duplicated().sum())
print("Duplicated BookingID  :", raw["BookingID"].duplicated().sum())

### 2.4 Categorical columns: inconsistent labels

In [ ]:
for col in ["Status", "PaymentMethod"]:
    print(f"--- {col} ---")
    print(raw[col].value_counts(dropna=False))
    print()

`Status` has **9 spellings for 2 real outcomes** (case differences, `Canceled` vs `Cancelled`, trailing spaces). `PaymentMethod` is already consistent.

### 2.5 Numeric columns

In [ ]:
raw[["Adults", "Children", "UnitPriceUSD", "DiscountPct"]].describe().round(2)

In [ ]:
print("DiscountPct values:", sorted(raw["DiscountPct"].dropna().unique().tolist()))
print("Children values    :", sorted(raw["Children"].unique().tolist()))
print("Adults values      :", sorted(raw["Adults"].unique().tolist()))

### 2.6 Integrity checks

In [ ]:
# ID formats
print("BookingID format valid :", raw["BookingID"].str.match(r"^BK-2023-\d{5}$").all())
print("CustomerID format valid:", raw["CustomerID"].str.match(r"^C\d{5}$").all())

# Dates
booking = pd.to_datetime(raw["BookingDate"], errors="coerce")
travel = pd.to_datetime(raw["TravelStartDate"], errors="coerce")
print("Unparseable BookingDate :", booking.isna().sum())
print("Unparseable TravelStart :", travel.isna().sum())
print("Travel before booking   :", (travel < booking).sum())
print("Booking date range      :", booking.min().date(), "to", booking.max().date())
print("Travel date range       :", travel.min().date(), "to", travel.max().date())

In [ ]:
# Package pricing: how many different unit prices does each package have?
raw.groupby("PackageID")["UnitPriceUSD"].nunique().describe()

### 2.7 Screening summary

| Finding | Detail | Action |
|---|---|---|
| Inconsistent `Status` labels | 9 spellings for 2 outcomes | Standardise |
| Missing `AgentID` | 1,100 rows (28.4%) | Keep as null |
| Missing `DiscountPct` | 120 rows (3.1%) | Keep as null |
| Duplicates | None | None needed |
| Date validity | All parse; no travel date before its booking date | Reformat only |
| ID formats | All valid | None needed |

## 3. Data cleaning

### 3.1 Standardise `Status`

In [ ]:
df = raw.copy()

status_map = {"completed": "Completed", "cancelled": "Cancelled"}

df["Status"] = (
    df["Status"]
    .str.strip()
    .str.lower()
    .replace({"canceled": "cancelled"})
    .map(status_map)
)

assert df["Status"].notna().all(), "Unmapped Status value found"
df["Status"].value_counts()

### 3.2 Rename `Adults` to `AdultCount`

In [ ]:
df = df.rename(columns={"Adults": "AdultCount"})
df.columns.tolist()

### 3.3 Reformat dates to DD/MM/YYYY

In [ ]:
for col in ["BookingDate", "TravelStartDate"]:
    df[col] = pd.to_datetime(df[col], format="%Y-%m-%d").dt.strftime("%d/%m/%Y")

df[["BookingDate", "TravelStartDate"]].head()

### 3.4 Missing values

`AgentID` and `DiscountPct` are **left as null** rather than imputed, because the true values are unknown. A blank agent may mean "direct booking" and a blank discount may mean "no discount", but this should be confirmed with the business before filling them in.

## 4. Data transformation (new columns)

- `HeadCount` = `AdultCount` + `Children`
- `TotalPrice` = `HeadCount` x `UnitPriceUSD`
- `ActualPaidPrice` = `TotalPrice` after discount

> **Note on the discount formula.** `DiscountPct` contains fractions (0, 0.05, 0.10, 0.15). The existing cleaned file applies it as `TotalPrice * (1 - DiscountPct / 100)`, so a "15%" discount only reduces the price by 0.15%. If the discount is meant to be 15%, set `DISCOUNT_AS_FRACTION = True` in the first cell. This changes `ActualPaidPrice` for every discounted booking, so confirm the intended meaning first.

In [ ]:
df["HeadCount"] = df["AdultCount"] + df["Children"]
df["TotalPrice"] = df["HeadCount"] * df["UnitPriceUSD"]

if DISCOUNT_AS_FRACTION:
    df["ActualPaidPrice"] = df["TotalPrice"] * (1 - df["DiscountPct"])
else:
    df["ActualPaidPrice"] = df["TotalPrice"] * (1 - df["DiscountPct"] / 100)

df[["BookingID", "HeadCount", "UnitPriceUSD", "TotalPrice", "DiscountPct", "ActualPaidPrice"]].head()

## 5. Validation of the cleaned data

In [ ]:
checks = {
    "Row count matches raw": len(df) == len(raw),
    "BookingID unique": df["BookingID"].is_unique,
    "Status has exactly 2 values": set(df["Status"].unique()) == {"Completed", "Cancelled"},
    "HeadCount = AdultCount + Children": (df["HeadCount"] == df["AdultCount"] + df["Children"]).all(),
    "TotalPrice = HeadCount x UnitPriceUSD": (df["TotalPrice"] == df["HeadCount"] * df["UnitPriceUSD"]).all(),
    "PaymentMethod unchanged": df["PaymentMethod"].equals(raw["PaymentMethod"]),
    "ActualPaidPrice null only where DiscountPct null": df["ActualPaidPrice"].isna().equals(df["DiscountPct"].isna()),
}

for name, ok in checks.items():
    print(("PASS  " if ok else "FAIL  ") + name)

assert all(checks.values()), "One or more validation checks failed"
print()
print("Remaining nulls:")
print(df.isna().sum()[lambda s: s > 0])

### Things to review before using the data

- Cancelled bookings still carry an `ActualPaidPrice`. Decide whether this should be zero or refund-adjusted before analysing revenue.

In [ ]:
df.groupby("Status")["ActualPaidPrice"].agg(["count", "sum"]).round(0)

## 6. Export the cleaned file

In [ ]:
df.to_csv(CLEAN_PATH, index=False)
print(f"Saved {len(df):,} rows x {df.shape[1]} columns to {CLEAN_PATH}")

In [ ]:
# Quick reload check
check = pd.read_csv(CLEAN_PATH)
print(check.shape)
check.head()

## Next steps

- Apply the same screening and cleaning to the 2024 and 2025 raw files
- Add `LeadTimeDays` (travel start minus booking date) and `BookingMonth`
- Analyse cancellation rate by package, agent and payment method